In [2]:
# COS40007 Artificial Intelligence for Engineering
# Portfolio Assessment 3: "Develop an AI model by your own decision"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import pickle
import warnings
warnings.filterwarnings('ignore')

#---------------------------------------------------------------------------
# Step 1: Data Preparation
#---------------------------------------------------------------------------

# Load the dataset
df = pd.read_csv('vegemite.csv')
print(f"Dataset loaded with shape: {df.shape}")

# Initial data exploration
print("\nFirst 5 rows:")
print(df.head())

print("\nData types and basic information:")
print(df.info())

print("\nStatistical summary:")
print(df.describe())

# Check class distribution before shuffling
print("\nInitial class distribution:")
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True).round(4) * 100, '%')

# 1. Shuffle the dataset
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("\nDataset shuffled.")

# 2. Randomly take out 1000 data points with balanced classes
test_samples = []
target_per_class = 1000 // 3  # Aim for ~333 samples per class

# Select samples for each class
for class_val in df_shuffled['Class'].unique():
    class_df = df_shuffled[df_shuffled['Class'] == class_val]
    samples_needed = min(len(class_df), target_per_class)
    selected_samples = class_df.sample(n=samples_needed, random_state=42)
    test_samples.append(selected_samples)
    print(f"Selected {len(selected_samples)} samples for class {class_val}")
    
    # Remove these samples from the shuffled dataframe to avoid duplication
    df_shuffled = df_shuffled.loc[~df_shuffled.index.isin(selected_samples.index)]

# Combine all test samples
test_df = pd.concat(test_samples).reset_index(drop=True)

# If we don't have exactly 1000 samples, adjust as needed
remaining = 1000 - len(test_df)
if remaining > 0:
    print(f"Need {remaining} more samples to reach 1000")
    additional_samples = df_shuffled.sample(n=remaining, random_state=42)
    test_df = pd.concat([test_df, additional_samples]).reset_index(drop=True)

# The remaining data becomes the training set
train_df = df_shuffled.reset_index(drop=True)

print(f"\nTest dataset shape: {test_df.shape}")
print(f"Test dataset class distribution:")
print(test_df['Class'].value_counts().sort_index())

print(f"\nTraining dataset shape: {train_df.shape}")
print(f"Training dataset class distribution:")
print(train_df['Class'].value_counts().sort_index())

# For feature construction, we need to fix potential issues in the dataset

# 1. Check for constant value columns
constant_cols = [col for col in train_df.columns if train_df[col].nunique() == 1]
print(f"\nConstant value columns: {constant_cols}")

if constant_cols:
    # Remove constant columns
    train_df = train_df.drop(columns=constant_cols)
    test_df = test_df.drop(columns=constant_cols)
    print(f"Removed {len(constant_cols)} constant columns. New train shape: {train_df.shape}")

# 2. Check for columns with few integer values
few_integer_cols = []
for col in train_df.columns:
    if col != 'Class' and train_df[col].nunique() <= 10 and pd.api.types.is_numeric_dtype(train_df[col]):
        few_integer_cols.append(col)
        print(f"Column {col} has {train_df[col].nunique()} unique values: {sorted(train_df[col].unique())}")

# Convert columns with few integer values to categorical
for col in few_integer_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')
    print(f"Converted {col} to categorical")

# 3. Check class balance
class_counts = train_df['Class'].value_counts()
print("\nClass distribution in training data:")
print(class_counts)

# Calculate class imbalance ratio (smallest to largest)
imbalance_ratio = class_counts.min() / class_counts.max()
print(f"Class imbalance ratio: {imbalance_ratio:.4f}")

# If the imbalance ratio is less than 0.8 (20% difference), apply resampling
if imbalance_ratio < 0.8:
    print("Class imbalance detected. Applying resampling...")
    
    # Separate features and target
    X_train = train_df.drop('Class', axis=1)
    y_train = train_df['Class']
    
    # If a class is significantly overrepresented, use undersampling
    majority_class = class_counts.idxmax()
    if class_counts[majority_class] > 1.5 * class_counts.median():
        print(f"Majority class {majority_class} is significantly overrepresented. Using undersampling.")
        
        # Calculate target counts for undersampling
        target_counts = {c: min(class_counts[c], int(1.2 * class_counts.median())) for c in class_counts.index}
        rus = RandomUnderSampler(sampling_strategy=target_counts, random_state=42)
        X_resampled, y_resampled = rus.fit_resample(X_train, y_train)
        
    # For general imbalance, use SMOTE for oversampling
    else:
        print("Using SMOTE to create synthetic samples for minority classes.")
        
        # Apply SMOTE to balance classes
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
    
    # Create a new balanced DataFrame
    train_df_balanced = pd.DataFrame(X_resampled, columns=X_train.columns)
    train_df_balanced['Class'] = y_resampled
    
    print(f"After resampling - Training data shape: {train_df_balanced.shape}")
    print(f"After resampling - Class distribution:")
    print(train_df_balanced['Class'].value_counts().sort_index())
    
    # Use the balanced dataset for training
    train_df = train_df_balanced
else:
    print("Classes are relatively balanced. No resampling needed.")

# 4. Look for composite features through exploration
# Correlation analysis to find relationships
corr = train_df.select_dtypes(include=[np.number]).corr()

# Visualize correlation matrix
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()

# Identify SP (set point) features and PV (process variables) features
sp_features = [col for col in train_df.columns if col.endswith('SP')]
pv_features = [col for col in train_df.columns if col.endswith('PV')]

print(f"\nSet Point features ({len(sp_features)}):", sp_features)
print(f"Process Variable features ({len(pv_features)}):", pv_features)

# Create SP-PV difference features for corresponding variables
composite_features = []

for sp in sp_features:
    base_name = sp[:-3]  # Remove the 'SP' suffix
    corresponding_pv = base_name + 'PV'
    
    if corresponding_pv in pv_features:
        feature_name = f"Diff_{base_name.replace(' ', '_')}"
        train_df[feature_name] = train_df[sp] - train_df[corresponding_pv]
        test_df[feature_name] = test_df[sp] - test_df[corresponding_pv]
        composite_features.append(feature_name)
        print(f"Created composite feature: {feature_name}")

# Also create ratios for interesting pairs
for sp in sp_features:
    for other_sp in sp_features:
        if sp != other_sp and np.abs(corr.get(sp, {}).get(other_sp, 0)) > 0.5:
            feature_name = f"Ratio_{sp.replace(' ', '_')}_{other_sp.replace(' ', '_')}"
            train_df[feature_name] = train_df[sp] / (train_df[other_sp] + 1e-5)  # Avoid division by zero
            test_df[feature_name] = test_df[sp] / (test_df[other_sp] + 1e-5)
            composite_features.append(feature_name)
            print(f"Created ratio feature: {feature_name}")

print(f"\nCreated {len(composite_features)} new composite features")

# 5. Final feature count
print(f"\nFinal feature count: {train_df.shape[1] - 1}")  # Excluding Class

# Save the prepared datasets
train_df.to_csv('vegemite_train_prepared.csv', index=False)
test_df.to_csv('vegemite_test_prepared.csv', index=False)

#---------------------------------------------------------------------------
# Step 2: Feature selection, Model Training and Evaluation
#---------------------------------------------------------------------------

# 6. Feature selection
X_train = train_df.drop('Class', axis=1)
y_train = train_df['Class']
X_test = test_df.drop('Class', axis=1)
y_test = test_df['Class']

# Convert categorical columns to numeric for feature selection
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Ensure X_train and X_test have the same columns
missing_cols = set(X_train.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0
X_test = X_test[X_train.columns]

# Determine if we need all features by looking at feature importance
print("\nChecking feature importance...")
selector = SelectKBest(f_classif, k='all')
selector.fit(X_train, y_train)

# Get feature scores
feature_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': selector.scores_
})
feature_scores = feature_scores.sort_values('Score', ascending=False)

print("\nTop 20 features by importance:")
print(feature_scores.head(20))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_20_features = feature_scores.head(20)
sns.barplot(x='Score', y='Feature', data=top_20_features)
plt.title('Top 20 Features by Importance')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

# Calculate cumulative importance to guide feature selection
feature_scores['Cumulative_Importance'] = feature_scores['Score'].cumsum() / feature_scores['Score'].sum()
print("\nCumulative importance of features:")
print(feature_scores[['Feature', 'Score', 'Cumulative_Importance']].head(20))

# Select features that collectively account for 95% of importance
important_features = feature_scores[feature_scores['Cumulative_Importance'] <= 0.95]['Feature'].tolist()
print(f"\nSelected {len(important_features)} features that account for 95% of importance")

X_train_selected = X_train[important_features]
X_test_selected = X_test[important_features]

# 7. Train multiple ML models
print("\n7. Training multiple ML models with selected features...")

# Define models to train
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, multi_class='multinomial')
}

# Dictionary to store results
results = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train_selected, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_selected)
    
    # 8. Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    # Print results
    print(f"{name} - Accuracy: {accuracy:.4f}")
    print(f"{name} - Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1-score': report['macro avg']['f1-score'],
        'confusion_matrix': conf_matrix
    }
    
    # Visualize confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig(f'confusion_matrix_{name.lower().replace(" ", "_")}.png')
    plt.close()

# 9. Compare models
print("\n9. Comparing models across different evaluation measures:")

# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[model]['accuracy'] for model in results],
    'Precision': [results[model]['precision'] for model in results],
    'Recall': [results[model]['recall'] for model in results],
    'F1-Score': [results[model]['f1-score'] for model in results]
})

print(comparison.sort_values('F1-Score', ascending=False))

# Visualize model comparison
plt.figure(figsize=(12, 6))
comparison_melt = pd.melt(comparison, id_vars=['Model'], 
                         value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                         var_name='Metric', value_name='Score')
sns.barplot(x='Model', y='Score', hue='Metric', data=comparison_melt)
plt.title('Model Comparison')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('model_comparison.png')
plt.close()

# 10. Select the best-performing model as AI
# We'll choose based on F1-score as it balances precision and recall
best_model_name = comparison.sort_values('F1-Score', ascending=False).iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f"\n10. Selected best model: {best_model_name}")
print(f"Justification: {best_model_name} achieved the highest F1-Score of {comparison[comparison['Model']==best_model_name]['F1-Score'].values[0]:.4f}, "
      f"which balances precision and recall. This model provides the best overall performance across all classes.")

# 11. Save the selected model
print("\n11. Saving the best model...")
with open(f'best_model_{best_model_name.replace(" ", "_").lower()}.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save selected features list for future use
with open('selected_features.pkl', 'wb') as f:
    pickle.dump(important_features, f)

print(f"Model saved as best_model_{best_model_name.replace(' ', '_').lower()}.pkl")
print("Selected features saved as selected_features.pkl")

#---------------------------------------------------------------------------
# Step 3: ML to AI
#---------------------------------------------------------------------------

print("\nStep 3: ML to AI")

# 12. We already separated 1000 rows for testing at the beginning
print("12. Using the 1000 rows set aside earlier for testing")

# 13. Load the model
print("\n13. Loading the saved model...")
with open(f'best_model_{best_model_name.replace(" ", "_").lower()}.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('selected_features.pkl', 'rb') as f:
    selected_features = pickle.load(f)

print(f"Loaded model: {best_model_name}")
print(f"Loaded {len(selected_features)} selected features")

# 14. Format the test data with the same features as training
print("\n14. Preparing test data with the same features as training...")
X_test_full = test_df.drop('Class', axis=1)
y_test_full = test_df['Class']

# Convert to the same format as training
X_test_encoded = pd.get_dummies(X_test_full, drop_first=True)

# Ensure all selected features are present
missing_cols = set(selected_features) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0

# Select only the features used in training
X_test_prepared = X_test_encoded[selected_features]

# 15. Make predictions using the loaded model
print("\n15. Making predictions with the loaded model...")
y_pred_full = loaded_model.predict(X_test_prepared)

# 16. Measure performance on the test set
print("\n16. Measuring performance on 1000 unseen data points...")
accuracy = accuracy_score(y_test_full, y_pred_full)
conf_matrix = confusion_matrix(y_test_full, y_pred_full)
report = classification_report(y_test_full, y_pred_full)

print(f"Accuracy on test set: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

print("\nConfusion Matrix:")
print(conf_matrix)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name} on Test Data')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('test_confusion_matrix.png')
plt.close()

# 17. Evaluate all models on the test set
print("\n17. Measuring performance of all models on test data...")

test_results = {}
for name, model_info in results.items():
    model = model_info['model']
    y_pred = model.predict(X_test_prepared)
    
    accuracy = accuracy_score(y_test_full, y_pred)
    report = classification_report(y_test_full, y_pred, output_dict=True)
    
    test_results[name] = {
        'accuracy': accuracy,
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1-score': report['macro avg']['f1-score']
    }
    
    print(f"\n{name} - Test Accuracy: {accuracy:.4f}")
    print(f"{name} - Test F1-Score: {report['macro avg']['f1-score']:.4f}")

# Create comparison dataframe for test results
test_comparison = pd.DataFrame({
    'Model': list(test_results.keys()),
    'Test Accuracy': [test_results[model]['accuracy'] for model in test_results],
    'Test Precision': [test_results[model]['precision'] for model in test_results],
    'Test Recall': [test_results[model]['recall'] for model in test_results],
    'Test F1-Score': [test_results[model]['f1-score'] for model in test_results]
})

# Compare with training results
print("\nModel comparison on test data (sorted by F1-Score):")
print(test_comparison.sort_values('Test F1-Score', ascending=False))

# Check if our model selection remains the same
best_test_model = test_comparison.sort_values('Test F1-Score', ascending=False).iloc[0]['Model']
print(f"\nBest model on test data: {best_test_model}")
print(f"Original best model: {best_model_name}")

if best_test_model == best_model_name:
    print("The same model performs best on both training and test data, confirming our selection.")
else:
    print("Different models performed best on training vs. test data. This suggests possible overfitting or differences in data distribution.")

#---------------------------------------------------------------------------
# Step 4: Develop rules from the ML model
#---------------------------------------------------------------------------

print("\nStep 4: Develop rules from the ML model")

# Filter features to keep only SP features
sp_only_df = train_df[sp_features + ['Class']]
print(f"Filtered dataset to include only {len(sp_features)} SP features")
print(sp_only_df.head())

# Train a decision tree model using only SP features
X_sp = sp_only_df.drop('Class', axis=1)
y_sp = sp_only_df['Class']

# Train a decision tree with limited depth for interpretability
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_sp, y_sp)

# Print the tree using export_text
tree_rules = export_text(dt_model, feature_names=list(X_sp.columns))
print("\nDecision Tree Rules:")
print(tree_rules)

# Extract rules from the decision tree
def get_rules(tree, feature_names, class_names):
    tree_ = tree.tree_
    
    def recurse(node, depth, parent, rules, path=[]):
        if tree_.feature[node] != -2:  # Not a leaf node
            name = feature_names[tree_.feature[node]]
            threshold = tree_.threshold[node]
            
            # Left branch - less than or equal to threshold
            path_left = path + [(name, "<=", threshold)]
            recurse(tree_.children_left[node], depth + 1, node, rules, path_left)
            
            # Right branch - greater than threshold
            path_right = path + [(name, ">", threshold)]
            recurse(tree_.children_right[node], depth + 1, node, rules, path_right)
        else:  # Leaf node
            class_val = np.argmax(tree_.value[node][0])
            if path:  # Avoid empty paths
                rules.append((path, class_names[class_val], tree_.value[node][0][class_val] / sum(tree_.value[node][0])))
    
    rules = []
    recurse(0, 1, -1, rules)
    return rules

# Extract rules from the tree
rules = get_rules(dt_model, list(X_sp.columns), ['Class 0', 'Class 1', 'Class 2'])

# Print formatted rules
print("\nExtracted Set Point Rules:")
for i, (path, outcome, confidence) in enumerate(rules):
    # Format the rule text
    rule_text = " AND ".join([f"{feature} {op} {threshold:.2f}" for feature, op, threshold in path])
    print(f"Rule {i+1}: IF {rule_text} THEN {outcome} (confidence: {confidence:.2f})")

# Save the decision tree model
with open('sp_decision_tree.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

print("\nSaved decision tree model as sp_decision_tree.pkl")

# Create visualizations of rules for SP features
# For each class, show the range of values for important SP features
plt.figure(figsize=(14, 10))
for i, feature in enumerate(X_sp.columns[:min(6, len(X_sp.columns))]):
    plt.subplot(3, 2, i+1)
    sns.boxplot(x='Class', y=feature, data=sp_only_df)
    plt.title(f'Distribution of {feature} by Class')
plt.tight_layout()
plt.savefig('sp_feature_distributions.png')
plt.close()

print("\nAnalysis complete. All steps successfully executed.")

Dataset loaded with shape: (15237, 47)

First 5 rows:
   FFTE Feed tank level SP  FFTE Production solids SP  FFTE Steam pressure SP  \
0                     50.0                      40.74                   125.0   
1                     50.0                      40.74                   125.0   
2                     50.0                      40.74                   125.0   
3                     50.0                      40.74                   125.0   
4                     50.0                      39.00                    90.0   

   TFE Out flow SP  TFE Production solids SP  TFE Vacuum pressure SP  \
0          2897.65                      69.0                  -80.00   
1          2897.65                      69.0                  -79.45   
2          2897.65                      69.0                  -71.54   
3          2897.65                      69.0                  -68.44   
4          2694.62                      64.0                  -80.00   

   TFE Steam pressure SP  